
# Experiment 02B — Final-Configuration Architectural Ablation

Notebook ini **tidak mengganti Experiment 02 ablation lama**. Experiment 02 lama tetap digunakan sebagai diagnostic terhadap **HMRNet-Full-Initial**.

Notebook ini menambahkan ablation dengan **HMRNet-Final sebagai reference configuration**:

1. **HMRNet-Final**
2. **HMRNet-Final − CRA**
3. **HMRNet-Final − AF** — Adaptive Fusion diganti dengan **uniform averaging**
4. **HMRNet-Final − CRA − AF**

Opsional:
- HMRNet-Final − AL
- HMRNet-Final − DL
- HMRNet-Final-Concat

`fusion_mode="concat"` pada implementasi HMRNet adalah **concatenation-based fusion** karena hasil concatenation masih diproyeksikan melalui layer learnable.


## 1. Mount Google Drive

In [ ]:

from google.colab import drive
drive.mount("/content/drive")


## 1A. Install required dependencies

Jalankan cell ini **setelah mount Google Drive dan sebelum import modul project**.  
`gensim==4.4.0` diperlukan oleh `src/topic_metrics.py` untuk menghitung `C_v`, `C_NPMI`, dan UMass.


In [ ]:
# Install only the dependencies required by the evaluation code.
# Safe for the existing HMRNet Colab environment.
%pip install -q "gensim==4.4.0" openpyxl

import gensim
print("gensim version:", gensim.__version__)


## 2. Imports and project paths

In [ ]:
# Dependency guard: if this fails, run the dependency-install cell above first.
try:
    import gensim
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "gensim belum terpasang. Jalankan cell 'Install required dependencies' terlebih dahulu."
    ) from exc


from pathlib import Path
import sys
import gc
import copy
import json
import random
import re
import hashlib
import warnings

import joblib
import numpy as np
import pandas as pd
import torch

from IPython.display import display

PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/UDB/Riset/2026"),
]

PROJECT_ROOT = None
for candidate in PROJECT_CANDIDATES:
    if candidate.exists() and (candidate / "src" / "hmrnet.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project HMRNet tidak ditemukan. Perbaiki PROJECT_CANDIDATES sesuai lokasi Drive."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import load_feature_bundle
from src.hmrnet import HMRNet, HMRNetConfig
from src.topic_metrics import evaluate_topics
from src.stability_metrics import evaluate_stability_runs

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

ABLATION_ROOT = CHECKPOINTS_DIR / "final_architecture_ablation"
ABLATION_MODEL_DIR = ABLATION_ROOT / "models"
ABLATION_RESULT_DIR = ABLATION_ROOT / "results"

for d in [
    RESULTS_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
    ABLATION_ROOT,
    ABLATION_MODEL_DIR,
    ABLATION_RESULT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Load the fixed feature bundle

In [ ]:

bundle = load_feature_bundle(FEATURE_DIR)

raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

n_documents = len(processed_texts)

assert bow.shape[0] == n_documents
assert semantic_embeddings.shape[0] == n_documents
assert document_embeddings.shape[0] == n_documents

print("Documents:", n_documents)
print("Vocabulary:", len(vocabulary))
print("BoW:", bow.shape)
print("Semantic embeddings:", semantic_embeddings.shape)
print("Document embeddings:", document_embeddings.shape)

if "bow_normalized" in bundle:
    print("bow_normalized tersedia, tetapi eksperimen ini mempertahankan raw BoW agar konsisten dengan Stage 8 v2.")


## 4. Recover and lock the HMRNet-Final configuration

In [ ]:

BEST_CONFIG_PATH = TABLES_DIR / "experiment_08v2_best_config.json"
FINAL_CHECKPOINT = CHECKPOINTS_DIR / "HMRNet-Stage8-v2-CV-GT070.pt"

REQUIRED_FINAL_SETTINGS = {
    "n_topics": 20,
    "hidden_dim": 256,
    "attention_heads": 4,
    "dropout": 0.20,
    "learning_rate": 0.002,
    "weight_decay": 1e-5,
    "batch_size": 128,
    "epochs": 60,
    "patience": 16,
    "kl_weight": 0.10,
    "alignment_weight": 0.10,
    "diversity_weight": 0.07,
    "stability_weight": 0.0,
    "use_lexical": True,
    "use_semantic": True,
    "use_document": True,
    "use_cra": True,
    "fusion_mode": "adaptive",
    "use_alignment_loss": True,
    "use_diversity_loss": True,
    "use_stability_loss": False,
    "branch_noise_std": 0.0,
    "gradient_clip": 5.0,
}

if FINAL_CHECKPOINT.exists():
    tmp_model = HMRNet.load(FINAL_CHECKPOINT, device=DEVICE)
    FINAL_BASE_CONFIG = copy.deepcopy(tmp_model.config)
    del tmp_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Final configuration source: checkpoint")
else:
    FINAL_BASE_CONFIG = HMRNetConfig()
    print("Final checkpoint not found; using HMRNetConfig as base.")

if BEST_CONFIG_PATH.exists():
    with open(BEST_CONFIG_PATH, "r", encoding="utf-8") as f:
        best_payload = json.load(f)

    selected = best_payload.get("selected_config", {})
    for key, value in selected.items():
        if hasattr(FINAL_BASE_CONFIG, key):
            setattr(FINAL_BASE_CONFIG, key, value)

    if "selected_epochs" in best_payload and hasattr(FINAL_BASE_CONFIG, "epochs"):
        FINAL_BASE_CONFIG.epochs = int(best_payload["selected_epochs"])

    print("Stage 8 v2 JSON configuration applied.")

# Lock manuscript-final settings explicitly.
for key, value in REQUIRED_FINAL_SETTINGS.items():
    if not hasattr(FINAL_BASE_CONFIG, key):
        raise AttributeError(f"HMRNetConfig tidak memiliki field: {key}")
    setattr(FINAL_BASE_CONFIG, key, value)

FINAL_BASE_CONFIG.device = DEVICE

final_config_dict = {
    k: v for k, v in vars(FINAL_BASE_CONFIG).items()
    if not k.startswith("_")
}

audit_df = pd.DataFrame({
    "Parameter": list(REQUIRED_FINAL_SETTINGS.keys()),
    "Expected_Final": list(REQUIRED_FINAL_SETTINGS.values()),
    "Actual_Final": [final_config_dict[k] for k in REQUIRED_FINAL_SETTINGS],
})

audit_df["Match"] = (
    audit_df["Expected_Final"].astype(str)
    == audit_df["Actual_Final"].astype(str)
)

display(audit_df)

if not audit_df["Match"].all():
    raise RuntimeError("Final configuration audit gagal.")

print("✓ HMRNet-Final configuration locked and verified.")


## 5. Experimental settings

In [ ]:

TOP_N_WORDS = 10

# Untuk manuscript final gunakan 10 seeds.
# Untuk screening cepat sementara dapat diganti menjadi [1, 2, 3].
SEEDS = list(range(1, 11))

FORCE_RETRAIN = False

# Gunakan hasil Stage 8 v2 final yang sudah ada agar reference tidak dilatih ulang.
REUSE_EXISTING_FINAL = True

# Core experiment: paling penting untuk membuktikan CRA dan AF pada konfigurasi final.
RUN_CORE_ARCHITECTURE_ABLATIONS = True

# Optional, jika ingin tabel ablation final lebih lengkap.
RUN_OPTIONAL_LOSS_ABLATIONS = False
RUN_OPTIONAL_CONCAT = False

print("SEEDS:", SEEDS)
print("REUSE_EXISTING_FINAL:", REUSE_EXISTING_FINAL)
print("RUN_OPTIONAL_LOSS_ABLATIONS:", RUN_OPTIONAL_LOSS_ABLATIONS)
print("RUN_OPTIONAL_CONCAT:", RUN_OPTIONAL_CONCAT)


## 6. Define final-configuration ablation variants

In [ ]:

VARIANTS = {}

if RUN_CORE_ARCHITECTURE_ABLATIONS:
    VARIANTS.update({
        "HMRNet-Final-CRA": {
            "use_cra": False,
            "fusion_mode": "adaptive",
        },
        "HMRNet-Final-AF": {
            "use_cra": True,
            "fusion_mode": "average",
        },
        "HMRNet-Final-CRA-AF": {
            "use_cra": False,
            "fusion_mode": "average",
        },
    })

if RUN_OPTIONAL_LOSS_ABLATIONS:
    VARIANTS.update({
        "HMRNet-Final-AL": {
            "use_alignment_loss": False,
            "alignment_weight": 0.0,
        },
        "HMRNet-Final-DL": {
            "use_diversity_loss": False,
            "diversity_weight": 0.0,
        },
    })

if RUN_OPTIONAL_CONCAT:
    VARIANTS["HMRNet-Final-Concat"] = {
        "use_cra": True,
        "fusion_mode": "concat",
    }

preview = []
for model_name, overrides in VARIANTS.items():
    row = {"Model": model_name}
    row.update(overrides)
    preview.append(row)

display(pd.DataFrame(preview))

print(
    "Semua parameter yang tidak disebutkan diwarisi dari HMRNet-Final: "
    "KL=.10, AL=.10, DL=.07, SL=0, stability module disabled, "
    "branch noise=0, patience=16, lr=.002, dropout=.20, batch=128."
)


## 7. Training and evaluation helpers

In [ ]:

def sanitize_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def clone_final_config(overrides=None, seed=None):
    cfg = copy.deepcopy(FINAL_BASE_CONFIG)

    for key, value in (overrides or {}).items():
        if not hasattr(cfg, key):
            raise AttributeError(f"Config field tidak ditemukan: {key}")
        setattr(cfg, key, value)

    if seed is not None:
        cfg.random_state = int(seed)

    cfg.device = DEVICE
    return cfg


def config_hash(model_name, overrides, seed):
    payload = {
        "model": model_name,
        "overrides": overrides,
        "seed": int(seed),
        "base_final": REQUIRED_FINAL_SETTINGS,
    }
    text = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:10]


def scalar_metrics(metrics):
    out = {}
    for key, value in metrics.items():
        if isinstance(value, (str, int, float, np.integer, np.floating)):
            out[key] = value
    return out


def fusion_summary(model):
    try:
        weights = model.get_fusion_weights(
            bow=bow,
            semantic_embeddings=semantic_embeddings,
            document_embeddings=document_embeddings,
        )
    except Exception as exc:
        print("Fusion-weight summary skipped:", exc)
        return {}

    arr = np.asarray(weights)
    if arr.ndim != 2 or arr.shape[1] < 3:
        return {}

    avg = arr.mean(axis=0)
    return {
        "Fusion_Lexical": float(avg[0]),
        "Fusion_Semantic": float(avg[1]),
        "Fusion_Document": float(avg[2]),
    }


def train_one_variant(model_name, overrides, seed, force=False):
    tag = config_hash(model_name, overrides, seed)
    safe = sanitize_name(model_name)

    model_path = ABLATION_MODEL_DIR / f"{safe}_seed{seed}_{tag}.pt"
    result_path = ABLATION_RESULT_DIR / f"{safe}_seed{seed}_{tag}.joblib"

    if result_path.exists() and model_path.exists() and not force:
        print(f"[CACHE] {model_name} | seed={seed}")
        return joblib.load(result_path)

    print("\n" + "=" * 88)
    print("MODEL:", model_name)
    print("SEED :", seed)
    print("OVERRIDES:", json.dumps(overrides, indent=2))
    print("=" * 88)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    cfg = clone_final_config(overrides=overrides, seed=seed)
    model = HMRNet(cfg)

    model.fit(
        bow=bow,
        semantic_embeddings=semantic_embeddings,
        document_embeddings=document_embeddings,
        vocabulary=vocabulary,
        validation_fraction=0.10,
        verbose=True,
    )

    model.save(model_path)

    topics = model.get_topics(top_n=TOP_N_WORDS)
    beta = model.get_topic_word_distribution()

    metrics = evaluate_topics(
        model_name=model_name,
        topics=topics,
        processed_texts=processed_texts,
        top_n=TOP_N_WORDS,
    )

    row = {
        "Model": model_name,
        "Seed": int(seed),
        "use_cra": bool(cfg.use_cra),
        "fusion_mode": str(cfg.fusion_mode),
        "use_alignment_loss": bool(cfg.use_alignment_loss),
        "alignment_weight": float(cfg.alignment_weight),
        "use_diversity_loss": bool(cfg.use_diversity_loss),
        "diversity_weight": float(cfg.diversity_weight),
        "use_stability_loss": bool(cfg.use_stability_loss),
        "stability_weight": float(cfg.stability_weight),
        "branch_noise_std": float(cfg.branch_noise_std),
        "learning_rate": float(cfg.learning_rate),
        "dropout": float(cfg.dropout),
        "batch_size": int(cfg.batch_size),
        "patience": int(cfg.patience),
        "epochs": int(cfg.epochs),
        **scalar_metrics(metrics),
        "Model_Path": str(model_path),
    }

    # Hanya seed pertama untuk menghindari inference fusion-weight yang berulang.
    if seed == SEEDS[0]:
        row.update(fusion_summary(model))

    payload = {
        "row": row,
        "topics": topics,
        "beta": beta,
        "model_path": str(model_path),
        "overrides": overrides,
    }

    joblib.dump(payload, result_path)

    print(
        f"RESULT | C_v={row.get('C_v', np.nan):.4f} | "
        f"C_NPMI={row.get('C_NPMI', np.nan):.4f} | "
        f"TD={row.get('Topic_Diversity', np.nan):.4f} | "
        f"TR={row.get('Topic_Redundancy', np.nan):.6f}"
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return payload


def mean_metric(df, col):
    if col not in df.columns:
        return np.nan
    return float(pd.to_numeric(df[col], errors="coerce").mean())


def std_metric(df, col):
    if col not in df.columns:
        return np.nan
    return float(pd.to_numeric(df[col], errors="coerce").std(ddof=1))


## 8. Reuse existing HMRNet-Final 10-seed results

In [ ]:

EXISTING_FINAL_RUNS_PATH = TABLES_DIR / "experiment_08v2_final_10seed_runs.xlsx"
EXISTING_FINAL_SUMMARY_PATH = TABLES_DIR / "experiment_08v2_final_10seed_summary.xlsx"
EXISTING_FINAL_PAIRWISE_PATH = TABLES_DIR / "experiment_08v2_final_10seed_pairwise.csv"

reference_runs_df = pd.DataFrame()
reference_summary_df = pd.DataFrame()
reference_pairwise_df = pd.DataFrame()

can_reuse_final = (
    REUSE_EXISTING_FINAL
    and EXISTING_FINAL_RUNS_PATH.exists()
    and EXISTING_FINAL_SUMMARY_PATH.exists()
)

if can_reuse_final:
    reference_runs_df = pd.read_excel(EXISTING_FINAL_RUNS_PATH)
    reference_summary_raw = pd.read_excel(EXISTING_FINAL_SUMMARY_PATH)

    reference_runs_df["Model"] = "HMRNet-Final"

    raw = reference_summary_raw.iloc[0].to_dict()

    reference_summary_df = pd.DataFrame([{
        "Model": "HMRNet-Final",
        "N_Seeds": int(len(reference_runs_df)),
        "Mean_C_v": raw.get("Mean_C_v", np.nan),
        "Std_C_v": raw.get("Std_C_v", np.nan),
        "Mean_C_NPMI": raw.get("Mean_C_NPMI", np.nan),
        "Mean_UMass": raw.get("Mean_UMass", np.nan),
        "Mean_Topic_Diversity": raw.get("Mean_Topic_Diversity", np.nan),
        "Mean_Topic_Redundancy": raw.get("Mean_Topic_Redundancy", np.nan),
        "Mean_TS": raw.get("Mean_TS", np.nan),
        "Std_TS": raw.get("Std_TS", np.nan),
        "CI_Lower": raw.get("CI_Lower", raw.get("CI95_Lower", np.nan)),
        "CI_Upper": raw.get("CI_Upper", raw.get("CI95_Upper", np.nan)),
        "Reference_Source": str(EXISTING_FINAL_SUMMARY_PATH),
    }])

    if EXISTING_FINAL_PAIRWISE_PATH.exists():
        reference_pairwise_df = pd.read_csv(EXISTING_FINAL_PAIRWISE_PATH)
        reference_pairwise_df["Model"] = "HMRNet-Final"

    print("✓ Existing HMRNet-Final 10-seed results reused.")
    display(reference_summary_df.round(4))
else:
    print("Existing final results unavailable or reuse disabled.")
    print("HMRNet-Final will be trained together with the ablation variants.")
    VARIANTS = {"HMRNet-Final": {}, **VARIANTS}


## 9. Run final-configuration ablations

In [ ]:

all_run_rows = []
summary_rows = []
pairwise_frames = []

for model_name, overrides in VARIANTS.items():
    print("\n" + "#" * 96)
    print("FINAL-CONFIG ABLATION:", model_name)
    print("#" * 96)

    beta_by_seed = {}
    rows = []

    for seed in SEEDS:
        payload = train_one_variant(
            model_name=model_name,
            overrides=overrides,
            seed=seed,
            force=FORCE_RETRAIN,
        )

        row = dict(payload["row"])
        rows.append(row)
        all_run_rows.append(row)
        beta_by_seed[seed] = np.asarray(payload["beta"])

    runs_df = pd.DataFrame(rows)

    pairwise_df, stability = evaluate_stability_runs(
        beta_by_run=beta_by_seed,
        metric="js",
        confidence=0.95,
    )

    pairwise_df["Model"] = model_name
    pairwise_frames.append(pairwise_df)

    summary = {
        "Model": model_name,
        "N_Seeds": len(SEEDS),
        "Mean_C_v": mean_metric(runs_df, "C_v"),
        "Std_C_v": std_metric(runs_df, "C_v"),
        "Min_C_v": float(pd.to_numeric(runs_df["C_v"], errors="coerce").min()),
        "Max_C_v": float(pd.to_numeric(runs_df["C_v"], errors="coerce").max()),
        "Mean_C_NPMI": mean_metric(runs_df, "C_NPMI"),
        "Mean_UMass": mean_metric(runs_df, "UMass"),
        "Mean_Topic_Diversity": mean_metric(runs_df, "Topic_Diversity"),
        "Mean_Topic_Redundancy": mean_metric(runs_df, "Topic_Redundancy"),
        **stability,
    }

    cfg = clone_final_config(overrides=overrides, seed=SEEDS[0])
    summary.update({
        "use_cra": bool(cfg.use_cra),
        "fusion_mode": str(cfg.fusion_mode),
        "use_alignment_loss": bool(cfg.use_alignment_loss),
        "alignment_weight": float(cfg.alignment_weight),
        "use_diversity_loss": bool(cfg.use_diversity_loss),
        "diversity_weight": float(cfg.diversity_weight),
        "use_stability_loss": bool(cfg.use_stability_loss),
        "stability_weight": float(cfg.stability_weight),
        "branch_noise_std": float(cfg.branch_noise_std),
    })

    summary_rows.append(summary)

ablation_runs_df = pd.DataFrame(all_run_rows)
ablation_summary_df = pd.DataFrame(summary_rows)

display(ablation_summary_df.round(4))


## 10. Combine with HMRNet-Final and compute deltas

In [ ]:

combined_runs_df = pd.concat(
    [df for df in [reference_runs_df, ablation_runs_df] if not df.empty],
    ignore_index=True,
    sort=False,
)

combined_summary_df = pd.concat(
    [df for df in [reference_summary_df, ablation_summary_df] if not df.empty],
    ignore_index=True,
    sort=False,
)

pair_sources = []
if not reference_pairwise_df.empty:
    pair_sources.append(reference_pairwise_df)
pair_sources.extend(pairwise_frames)

combined_pairwise_df = (
    pd.concat(pair_sources, ignore_index=True, sort=False)
    if pair_sources else pd.DataFrame()
)

if "HMRNet-Final" not in set(combined_summary_df["Model"]):
    raise RuntimeError("HMRNet-Final reference tidak tersedia.")

final_row = combined_summary_df.loc[
    combined_summary_df["Model"] == "HMRNet-Final"
].iloc[0]

for metric in [
    "Mean_C_v",
    "Mean_C_NPMI",
    "Mean_Topic_Diversity",
    "Mean_Topic_Redundancy",
    "Mean_TS",
]:
    if metric in combined_summary_df.columns and pd.notna(final_row.get(metric, np.nan)):
        combined_summary_df[f"Delta_{metric}_vs_Final"] = (
            pd.to_numeric(combined_summary_df[metric], errors="coerce")
            - float(final_row[metric])
        )

order = {
    "HMRNet-Final": 0,
    "HMRNet-Final-CRA": 1,
    "HMRNet-Final-AF": 2,
    "HMRNet-Final-CRA-AF": 3,
    "HMRNet-Final-AL": 4,
    "HMRNet-Final-DL": 5,
    "HMRNet-Final-Concat": 6,
}

combined_summary_df["_order"] = (
    combined_summary_df["Model"].map(order).fillna(99)
)

combined_summary_df = (
    combined_summary_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

display_cols = [
    "Model",
    "Mean_C_v",
    "Std_C_v",
    "Mean_C_NPMI",
    "Mean_Topic_Diversity",
    "Mean_Topic_Redundancy",
    "Mean_TS",
    "Std_TS",
    "Delta_Mean_C_v_vs_Final",
    "Delta_Mean_TS_vs_Final",
]

display(
    combined_summary_df[
        [c for c in display_cols if c in combined_summary_df.columns]
    ].round(4)
)


## 11. Save outputs

In [ ]:

RUNS_OUT = TABLES_DIR / "experiment_02b_final_architecture_ablation_runs.xlsx"
SUMMARY_OUT = TABLES_DIR / "experiment_02b_final_architecture_ablation_summary.xlsx"
PAIRWISE_OUT = TABLES_DIR / "experiment_02b_final_architecture_ablation_pairwise.csv"
CONFIG_OUT = TABLES_DIR / "experiment_02b_final_architecture_ablation_config_audit.xlsx"

combined_runs_df.to_excel(RUNS_OUT, index=False)
combined_summary_df.to_excel(SUMMARY_OUT, index=False)
audit_df.to_excel(CONFIG_OUT, index=False)

if not combined_pairwise_df.empty:
    combined_pairwise_df.to_csv(PAIRWISE_OUT, index=False)

print("Saved:")
print("✓", RUNS_OUT)
print("✓", SUMMARY_OUT)
if not combined_pairwise_df.empty:
    print("✓", PAIRWISE_OUT)
print("✓", CONFIG_OUT)


## 12. Manuscript-ready table

In [ ]:

paper_table = combined_summary_df.rename(columns={
    "Mean_C_v": "C_v",
    "Mean_C_NPMI": "C_NPMI",
    "Mean_Topic_Diversity": "TD",
    "Mean_Topic_Redundancy": "TR",
    "Mean_TS": "TS",
})

paper_cols = [
    c for c in ["Model", "C_v", "C_NPMI", "TD", "TR", "TS"]
    if c in paper_table.columns
]

display(paper_table[paper_cols].round(4))

print(
    "\nInterpretation:\n"
    "1. Bandingkan CRA/AF variants terhadap HMRNet-Final, bukan HMRNet-Full-Initial.\n"
    "2. Nilai CRA/AF dari C_v, TD, TR, dan TS secara bersama, bukan C_v saja.\n"
    "3. HMRNet-Final-AF memakai uniform averaging sehingga benar-benar menghapus document-specific adaptive weighting.\n"
    "4. Pertahankan Experiment 02 lama sebagai initial-configuration diagnostic dan laporkan tabel ini secara terpisah."
)



## 13. Recommended manuscript positioning

Gunakan dua tabel ablation yang berbeda:

**Initial component ablation**
- HMRNet-Full-Initial
- −CRA, −AF, −AL, −DL, −SL, Concat

**Final-configuration architectural ablation**
- HMRNet-Final
- HMRNet-Final − CRA
- HMRNet-Final − AF
- HMRNet-Final − CRA − AF

Dengan desain ini, tabel pertama menjelaskan diagnosis konfigurasi awal, sedangkan tabel kedua menjawab secara langsung apakah CRA dan Adaptive Fusion berkontribusi setelah objective HMRNet dikalibrasi.
